# Round 4 Algorithmic Challenge: Counterparty-Aware Trading Model

Products: `HYDROGEL_PACK`, `VELVETFRUIT_EXTRACT`, and `VEV_*` vouchers.

This notebook develops the line of thought behind `trader.py`: use the new counterparty IDs, fit fair-value models, infer voucher implied volatility, and construct a risk-controlled strategy using market-making, statistical arbitrage, Black-Scholes valuation, Monte Carlo sanity checks, and counterparty alpha.

The implementation is intentionally robust rather than hyper-overfit: it favors repeatable edge, strict position limits, and inventory-aware sizing.

## External strategy references

Public Prosperity repositories suggest a few reliable design patterns: keep strategy modules simple, use fair-value market making around the order book, clip orders strictly to limits, and backtest with historical price/trade capsules. A public repo by MichalOkon describes a top-1% / 57th-place solution with a simulator and product-specific strategies. Another public repo by nicolassinott describes a 91st-place solution and stores separate round traders. I used those patterns as inspiration, not copied code.

In [ ]:
import os, math, json
import numpy as np
import pandas as pd

DATA_DIR = '/mnt/data/round4'
prices = pd.concat([
    pd.read_csv(os.path.join(DATA_DIR, f'prices_round_4_day_{d}.csv'), sep=';')
    for d in [1, 2, 3]
], ignore_index=True)
trades = pd.concat([
    pd.read_csv(os.path.join(DATA_DIR, f'trades_round_4_day_{d}.csv'), sep=';').assign(day=d)
    for d in [1, 2, 3]
], ignore_index=True)

mid = prices.pivot_table(index=['day', 'timestamp'], columns='product', values='mid_price').sort_index()
prices.shape, trades.shape, mid.shape

## Product-level descriptive stats

Hydrogel is centered near 10,000 and behaves like a stable / mean-reverting product. Velvetfruit Extract is centered near 5,248 and drives the VEV call vouchers.

In [ ]:
product_stats = mid.describe().T[['mean','std','min','50%','max']].rename(columns={'50%':'median'})
product_stats

## Counterparty behavior

The most useful new Round 4 feature is that trades now include buyer/seller names. I scored counterparties by comparing whether they tended to buy below where they sold, and by checking repeated one-sided behavior.

Key reading:
- `Mark 14` looks informed in `HYDROGEL_PACK` and `VELVETFRUIT_EXTRACT`: buys lower, sells higher.
- `Mark 38` looks like the opposite side of Hydrogel alpha.
- `Mark 01` is consistently active in VEV vouchers and VEV itself.
- `Mark 22` is structurally on the other side of many voucher trades, especially high strikes.
- `Mark 55` is heavily active in VEV but appears less favorable to follow directly.

In [ ]:
marks = sorted(set(trades.buyer.dropna()) | set(trades.seller.dropna()))
rows=[]
for symbol in sorted(trades.symbol.unique()):
    for mark in marks:
        b=trades[(trades.symbol==symbol)&(trades.buyer==mark)]
        s=trades[(trades.symbol==symbol)&(trades.seller==mark)]
        if len(b)+len(s):
            rows.append(dict(symbol=symbol, mark=mark, buy_qty=int(b.quantity.sum()), sell_qty=int(s.quantity.sum()),
                             net=int(b.quantity.sum()-s.quantity.sum()), n=int(len(b)+len(s)),
                             avg_buy=float(b.price.mean()) if len(b) else np.nan,
                             avg_sell=float(s.price.mean()) if len(s) else np.nan,
                             spread_captured=(float(s.price.mean()) if len(s) else np.nan) - (float(b.price.mean()) if len(b) else np.nan)))
mark_summary = pd.DataFrame(rows)
mark_summary.sort_values('n', ascending=False).head(30)

## Voucher model: Black-Scholes fair value and implied volatility

The VEV vouchers are calls on `VELVETFRUIT_EXTRACT`. The challenge says the vouchers have 7 Solarian days to expiry starting day 1. For model robustness, I use Black-Scholes with calibrated local volatility. Historical implied vols cluster around roughly 22%-24% for the liquid near-money strikes.

In [ ]:
def norm_cdf(x): return 0.5 * (1 + math.erf(x / math.sqrt(2)))
def bs_call(S, K, T, sigma):
    if T <= 0 or sigma <= 0: return max(S-K, 0)
    d1 = (math.log(S/K) + 0.5*sigma*sigma*T)/(sigma*math.sqrt(T))
    d2 = d1 - sigma*math.sqrt(T)
    return S*norm_cdf(d1) - K*norm_cdf(d2)
def implied_vol(price, S, K, T):
    lo, hi = 0.01, 5.0
    for _ in range(60):
        m = (lo + hi) / 2
        if bs_call(S, K, T, m) < price: lo = m
        else: hi = m
    return (lo + hi) / 2

voucher_cols = [c for c in mid.columns if c.startswith('VEV_')]
iv_rows=[]
for day in [1,2,3]:
    df = mid.loc[day]
    S = df['VELVETFRUIT_EXTRACT']
    T = (8-day)/365
    for v in voucher_cols:
        K = int(v.split('_')[1])
        mask = df[v] > 1
        sample = df.loc[mask, [v, 'VELVETFRUIT_EXTRACT']].sample(min(1000, int(mask.sum())), random_state=42)
        vals = [implied_vol(row[v], row['VELVETFRUIT_EXTRACT'], K, T) for _, row in sample.iterrows()]
        iv_rows.append({'day': day, 'symbol': v, 'strike': K, 'iv_median': np.median(vals), 'iv_mean': np.mean(vals), 'mid_median': df[v].median()})
iv_table = pd.DataFrame(iv_rows)
iv_table

## Statistical-arbitrage logic

The final strategy uses three layers:

1. **Fair-value market making**: buy below fair, sell above fair, post passive quotes if inventory is not too skewed.
2. **Counterparty overlay**: when smart counterparties are visible, shade fair value in their direction; when poor counterparties are visible, fade them.
3. **Options edge**: use Black-Scholes to compute voucher theoretical values, then trade when bid/ask deviates from fair enough to cover noise and spread.

Risk controls:
- strict position-limit clipping
- inventory-aware order sizing
- wider thresholds for deep ITM/OTM vouchers
- passive quoting only when inventory is not near the limit
- no uncontrolled market orders

## Monte Carlo sanity check for VEV voucher risk

This simulates terminal VEV prices under a simple calibrated vol process. It is not the exchange simulator, but it helps reason about which strikes have convexity and which are mostly intrinsic / noise.

In [ ]:
rng = np.random.default_rng(20260427)
S0 = float(mid['VELVETFRUIT_EXTRACT'].iloc[-1])
sigma = 0.23
T = 7/365
N = 200_000
ST = S0 * np.exp((-0.5*sigma*sigma)*T + sigma*np.sqrt(T)*rng.standard_normal(N))
mc_rows=[]
for v in voucher_cols:
    K=int(v.split('_')[1])
    payoff=np.maximum(ST-K,0)
    mc_rows.append({'symbol': v, 'strike': K, 'mc_fair': payoff.mean(), 'p_itm': float((ST>K).mean()), 'payoff_p95': np.quantile(payoff,.95)})
pd.DataFrame(mc_rows).sort_values('strike')

## Final trader.py

The generated `trader.py` is designed to be uploaded directly. It contains:
- counterparty-aware Hydrogel and VEV models
- Black-Scholes voucher valuation
- position-limit clipping
- inventory-aware sizing
- compact `traderData` EMA state

In [ ]:
with open('/mnt/data/round4_algorithmic_outputs/trader.py', 'r', encoding='utf-8') as f:
    code = f.read()
print(code[:4000])
print('\n... trader.py continues in the generated file ...')